In [4]:
import numpy as np
import math
import matplotlib.pyplot as plt
import time
import random
%matplotlib inline

In [5]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = Value(other) if isinstance(other, (int, float)) else other
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        
        return out
    
    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + other * -1

    def __rsub__(self, other):
        other = Value(other) if isinstance(other, (int, float)) else other
        return other - self

    def __mul__(self, other):
        other = Value(other) if isinstance(other, (int, float)) else other
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out
    
    def __rmul__(self, other):
        other = Value(other) if isinstance(other, (int, float)) else other
        return self * other
    
    def __pow__(self, x):
        other = Value(x) if isinstance(x, (int, float)) else x
        out = Value(self.data ** other.data, (self, other), '^')

        def _backward():
            self.grad += other.data * self.data ** (other.data-1)
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) -1)/(math.exp(2*x) +1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1-t**2) * out.grad
        out._backward = _backward
        
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
        

In [6]:
from graphviz import Digraph

def trace(root):
    #build a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # Left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        #for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape = 'record')
        if n._op:
            #if value is result of a operation create a op code for it
            dot.node(name = uid + n._op, label = n._op)
            #connect this to node
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        #connect n1 to the op of node 2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [ ]:
import os, shutil, sys
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]
print(sys.executable)
print(shutil.which("dot"))
print(os.environ.get("PATH"))

In [8]:
from dataclasses import dataclass, field

In [9]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w * x + b
        act = sum((wi * xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()   
        return out
    
    def parameters(self):
        return self.w + [self.b]

@dataclass(slots=True)
class Layer:
    nin: int
    nout: int
    neurons: list = field(init=False)
    
    def __post_init__(self):
        self.neurons = [Neuron(self.nin) for _ in range(self.nout)]


    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) ==1 else outs
    
    def parameters(self):
        return [p for neurons in self.neurons for p in neurons.parameters()]
    
@dataclass(slots=True)
class MLP:
    nin: int
    sizes: list

    layers: list = field(init=False)
    
    def __post_init__(self):
        #build the layers
        layout = [self.nin] + self.sizes
        self.layers = [Layer(layout[i], layout[i+1]) for i in range(len(self.sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [14]:
x = [2.0, 3.0, 4.2]
sizes = [4,4,1]
n = MLP(3, sizes)
params = n.parameters()
out = n(x)

In [15]:
xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, 1.0, -1.0]

In [16]:
h = 0.25
i = 0
iterations = 2000
start = time.time()
while i <= iterations:
     outs = [n(x) for x in xs]
     loss = sum((out-y)**2 for out, y in zip(outs, ys))
     for param in params:
         param.grad = 0.0

     loss.backward()
     for param in params:
         param.data -= param.grad * h 
     i += 1
     #h *= (iterations-i)/iterations
print("Loss: ", loss.data)
end = time.time()
print(f"Total time: {end-start:.2f} seconds")
    

Loss:  3.42333914629278e-05
Total time: 2.61 seconds
